## **Exercises**



7. When should you create a custom layer versus a custom model?

8. What are some use cases that require writing your own custom training loop?

9. Can custom Keras components contain arbitrary Python code, or must they be convertible to TF Functions?

10. What are the main rules to respect if you want a function to be convertible to a TF Function?

11. When would you need to create a dynamic Keras model? How do you do that? Why not make all your models dynamic?

12. Implement a custom layer that performs Layer Normalization (we will use this type of layer in Chapter 15):
    
    a. The build() method should define two trainable weights α and β, both of shape input_shape[-1:] and data type tf.float32. α should be initialized with 1s, and β with 0s.

    b. The call() method should compute the mean μ and standard deviation σ of each instance’s features. For this, you can use tf.nn.moments(inputs, axes=-1, keepdims=True), which returns the mean μ and the variance σ2 of all instances (compute the square root of the variance to get the standard deviation). Then the function should compute and return α⊗(X - μ)/(σ + ε) + β, where ⊗ represents itemwise multiplication (*) and ε is a smoothing term (small constant to avoid division by zero, e.g., 0.001).
    
    c. Ensure that your custom layer produces the same (or very nearly the same) output as the keras.layers.LayerNormalization layer.

13. Train a model using a custom training loop to tackle the Fashion MNIST dataset (see Chapter 10).

    a. Display the epoch, iteration, mean training loss, and mean accuracy over each epoch (updated at each iteration), as well as the validation loss and accuracy at the end of each epoch.
    
    b. Try using a different optimizer with a different learning rate for the upper layers and the lower layers.

# 1

### **How would you describe TensorFlow in a short sentence? What are its main features? Can you name other popular Deep Learning libraries?**

Tensorflow is a high level library for deep learning, and it utilize the GPU while training models and also provide high level modules to to train custom models other deep learning libraries is pytorch

# 2

### **Is TensorFlow a drop-in replacement for NumPy? What are the main differences between the two?**

TensorFlow and Numpy operation are similar to each other but the tensnorflow support the GPU and tensorflow create the Computation Graph, auto diffrentiation is a implemented in tensorflow and Tensorflow has more build in features than Numpy.

# 3

### **Do you get the same result with tf.range(10) and tf.constant(np.arange(10))?**

In [2]:
import tensorflow as tf
import numpy as np
print(tf.range(10))
print(tf.constant(np.arange(10)))

tf.Tensor([0 1 2 3 4 5 6 7 8 9], shape=(10,), dtype=int32)
tf.Tensor([0 1 2 3 4 5 6 7 8 9], shape=(10,), dtype=int64)


# 4

**Can you name six other data structures available in TensorFlow, beyond regular
tensors? A custom loss function can be defined by writing a function or by subclassing the
keras.losses.Loss class. When would you use each option?**

constant, variable, sparse, string, queue, set and Ragged and i will use Custom function if the problem is evaluate on different loss function

In [ ]:
import tensorflow as tf
import keras

class HuberLoss(keras.losses.Loss):
    def __init__(self, threshold=1.0, reduction="sum_over_batch_size", name="huber_loss"):
        super().__init__(reduction=reduction, name=name)
        self.threshold = threshold

    def call(self, y_true, y_pred):
        error = y_true - y_pred

        is_small_error = tf.abs(error) < self.threshold
        squared_loss = tf.square(error) / 2
        linear_loss = self.threshold * tf.abs(error) - self.threshold**2 / 2

        return tf.where(is_small_error, squared_loss, linear_loss)

    def get_config(self):
        config = super().get_config()
        config.update({
            "threshold": self.threshold
        })
        return config

# 6

### **Similarly, a custom metric can be defined in a function or a subclass of keras.metrics.Metric. When would you use each option?**

In [ ]:
import tensorflow as tf
from tensorflow import keras


class ThresholdAccuracy(keras.metrics.Metric):
    def __init__(self, threshold=0.01, name="threshold_accuracy", dtype=None):
        super().__init__(name=name, dtype=dtype)
        self.threshold = threshold
        
        self.total = self.add_weight(
            name="total",
            initializer="zeros"
        )
        
        self.correct = self.add_weight(
            name="correct",
            initializer="zeros"
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        error = tf.abs(y_true - y_pred)
        matches = tf.cast(error < self.threshold, self.dtype)

        if sample_weight is not None:
            sample_weight = tf.cast(sample_weight, self.dtype)
            matches *= sample_weight

        self.correct.assign_add(tf.reduce_sum(matches))
        self.total.assign_add(tf.cast(tf.size(matches), self.dtype))

    def result(self):
        return self.correct / self.total

    def reset_state(self):
        self.total.assign(0)
        self.correct.assign(0)

    def get_config(self):
        config = super().get_config()
        config.update({
            "threshold": self.threshold
        })
        return config